# Import Library

In [ ]:
import os
import re
import joblib
import numpy as np
import pandas as pd

from pathlib import Path

from nltk.tokenize import word_tokenize

# Config

In [ ]:
# =========================
# CONFIGURATION
# =========================

SCENARIO = 1

"""
1 : TF-IDF + Naive Bayes
2 : TF-IDF + SVM
3 : FastText + SVM
4 : IndoBERT
"""

# 0 = Full
# 1 = 10k
# 2 = 20k
# 3 = 30k
# 4 = 40k

CORPUS_SIZE = 1
CORPUS_MAP = {
    0:"full",
    1:"10k",
    2:"20k",
    3:"30k",
    4:"40k"
}
DATASET_NAME = CORPUS_MAP[CORPUS_SIZE]

print()

print("Scenario :", SCENARIO)
print("Corpus   :", DATASET_NAME)

In [ ]:
CONFIG = {

    1:{
        "name":"tfidf_nb",
        "vectorizer": f"models/skenario 1/tfidf_nb_vectorizer_{DATASET_NAME}.joblib",
        "model":f"models/skenario 1/tfidf_nb_{DATASET_NAME}.joblib"
    },

    2:{

        "name":"tfidf_svm",
        "vectorizer":f"models/skenario 2/tfidf_svm_vectorizer_{DATASET_NAME}.joblib",
        "model":f"models/skenario 2/tfidf_svm_{DATASET_NAME}.joblib"
    },

    3:{
        "name":"fasttext_svm",
        "embedding":f"models/skenario 3/fasttext_{DATASET_NAME}.joblib",
        "model":f"models/skenario 3/fasttext_svm_{DATASET_NAME}.joblib"
    },

    4:{
        "name":"indobert",
        "model_dir": f"models/skenario 4/indobert_{DATASET_NAME}"
    }

}

if SCENARIO not in CONFIG:
    raise ValueError("SCENARIO tidak valid!")

print()
print(CONFIG[SCENARIO])

# Load Data & Label Encoder

In [ ]:
df = pd.read_csv("dataset/unseen.csv")
print()

print(df.shape)
display(df.head())

In [ ]:
def preprocess_text(text):

    text = str(text).lower()

    text = re.sub(

        r"https?://\S+|www\.\S+",

        " ",

        text

    )

    text = re.sub(

        r"@\w+",

        " ",

        text

    )

    text = re.sub(

        r"#",

        "",

        text

    )

    text = re.sub(

        r"\s+",

        " ",

        text

    )

    return text.strip()

df["text_clean"] = df["text"].apply(preprocess_text)

In [ ]:
le = joblib.load("dataset/preprocessed/label_encoder.pkl")

print()
print("Preprocessing Completed!")

display(
    df[[
            "text",
            "text_clean"
        ]].head()
)

# Scenario 1

In [ ]:
if SCENARIO == 1:

    vectorizer = joblib.load(CONFIG[1]["vectorizer"])
    model = joblib.load(CONFIG[1]["model"])
    X = vectorizer.transform(df["text_clean"])
    pred = model.predict(X)

    print()
    print("TF-IDF + Naive Bayes Loaded!")

# Scenario 2

In [ ]:
if SCENARIO == 2:

    vectorizer = joblib.load(CONFIG[2]["vectorizer"])
    model = joblib.load(CONFIG[2]["model"])
    X = vectorizer.transform(df["text_clean"])
    pred = model.predict(X)

    print()
    print("TF-IDF + SVM Loaded!")

# Scenario 3

In [ ]:
def sentence_vector_fasttext(
    tokens,
    model,
    vector_size=300
):
    vectors = []
    for token in tokens:
        if token in model.wv:
            vectors.append(model.wv[token])

    if len(vectors) == 0:
        return np.zeros(vector_size)

    return np.mean(vectors, axis=0)

In [ ]:
if SCENARIO == 3:

    ft_model = joblib.load(CONFIG[3]["embedding"])
    svm_model = joblib.load(CONFIG[3]["model"])
    tokens = [
        word_tokenize(text)
        for text in df["text_clean"]
    ]

    X = np.array([
        sentence_vector_fasttext(
            token,
            ft_model
        )
        for token in tokens
    ])

    pred = svm_model.predict(X)

    print()
    print("FastText + SVM Loaded!")

# Scenario 4

In [ ]:
if SCENARIO == 4:

    import torch
    from transformers import (
        AutoTokenizer,
        AutoModelForSequenceClassification
    )

    MODEL_PATH = CONFIG[4]["model_dir"]
    tokenizer = AutoTokenizer.from_pretrained(MODEL_PATH)
    model = AutoModelForSequenceClassification.from_pretrained(MODEL_PATH)
    model.eval()

    pred = []
    for text in df["text_clean"]:
        inputs = tokenizer(
            text,
            return_tensors="pt",
            truncation=True,
            padding=True,
            max_length=128
        )

        with torch.no_grad():
            outputs = model(
                **inputs
            )

        label = torch.argmax(
            outputs.logits,
            dim=1
        )

        pred.append(label.item())

    pred = np.array(pred)

    print()
    print("IndoBERT Loaded!")

# Decode Prediction

In [ ]:
df["prediction"] = le.inverse_transform(pred)
print()
print("Prediction Completed!")

display(
    df[[
        "text",
        "prediction"
        ]]
)

# Result

In [ ]:
display(df)
print()
print("Total Data :", len(df))

Prediction Distirbution

In [ ]:
prediction_summary = (
    df["prediction"]
    .value_counts()
    .sort_index()
    .reset_index()
)

prediction_summary.columns = [
    "Category",
    "Count"
]

display(prediction_summary)

Save Prediction

In [ ]:
os.makedirs("results/prediction", exist_ok=True)

MODEL_NAME = CONFIG[SCENARIO]["name"]

OUTPUT = (
    f"results/prediction/"
    f"{MODEL_NAME}_{DATASET_NAME}.csv"
)

df.to_csv(OUTPUT, index=False)

print()
print("Prediction Saved!")

print()
print(OUTPUT)

Save summary

In [ ]:
SUMMARY_OUTPUT = (
    f"results/prediction/"
    f"{MODEL_NAME}_{DATASET_NAME}_summary.csv"
)

prediction_summary.to_csv(SUMMARY_OUTPUT, index=False)

print()
print("Summary Saved!")

print()
print(SUMMARY_OUTPUT)

Experiment Info

In [ ]:
print()
print("="*50)
print("PREDICTION FINISHED")
print("="*50)

print()
print("Scenario :")
print(CONFIG[SCENARIO]["name"])

print()
print("Corpus :")
print(DATASET_NAME)

print()
print("Input Data :")
print(len(df))

print()
print("Prediction File :")
print(OUTPUT)

print()
print("Summary File :")
print(SUMMARY_OUTPUT)

print()
print("="*50)